# Play Connect 4 against the trained agent

Loads a checkpoint from `checkpoints/c4/` (or any path you point at) and lets you play against it interactively. Works with a fully-trained model, a half-baked mid-training checkpoint, or even a randomly-initialised one if you just want to feel what raw MCTS-without-a-net does.

**Board layout.** Columns are indexed 0–6, left to right. You drop a piece in column `c` and it falls to the lowest empty row. Win by making 4-in-a-row horizontally, vertically, or diagonally.

In [1]:
%load_ext autoreload
%autoreload 2

import glob
import os

import numpy as np
import torch

from connect4 import Connect4
from mcts.mcts import MCTS
from model.model import ResNet

## 1. Load a checkpoint

By default this picks the highest-numbered `connect4_iter_*.pt` in `checkpoints/c4/`. To play against a specific iteration, set `CHECKPOINT_PATH`. To play against an untrained random net (useful as a sanity baseline), set `UNTRAINED = True`.

In [2]:
CHECKPOINT_DIR = "checkpoints/c4"
CHECKPOINT_PATH = None   # e.g. "checkpoints/c4/connect4_iter_0030.pt"
UNTRAINED = False        # True → fresh random model, no checkpoint load

# Model size must match the C4 preset in train/run_training.py
NUM_RES_BLOCKS = 3
NUM_HIDDEN = 64

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

game = Connect4()
model = ResNet(game, num_res_blocks=NUM_RES_BLOCKS, num_hidden=NUM_HIDDEN).to(device)

if UNTRAINED:
    print("Using untrained random-init model (no checkpoint).")
else:
    if CHECKPOINT_PATH is None:
        files = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "connect4_iter_*.pt")))
        if not files:
            raise FileNotFoundError(
                f"No connect4_iter_*.pt found in {CHECKPOINT_DIR}. "
                "Train first (`uv run python -m train.run_training --preset C4`), "
                "or set UNTRAINED = True above."
            )
        CHECKPOINT_PATH = files[-1]
    ckpt = torch.load(CHECKPOINT_PATH, weights_only=False, map_location=device)
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state_dict)
    iter_str = f" (iter {ckpt['iteration']})" if isinstance(ckpt, dict) and "iteration" in ckpt else ""
    print(f"Loaded {CHECKPOINT_PATH}{iter_str}")

model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}  |  params: {n_params:,}")

Loaded checkpoints/c4/connect4_iter_0050.pt (iter 50)
Device: mps  |  params: 236,143


## 2. MCTS settings

`NUM_SEARCHES` controls how hard the agent thinks per move. Cheap to crank up at play time — try 200, 500, or 1000 if you want a stronger opponent. Dirichlet noise is off (we want deterministic best-play here, not exploration).

In [15]:
NUM_SEARCHES = 50
MCTS_BATCH_SIZE = 20
C_PUCT = 1.0

def make_mcts():
    return MCTS(
        game,
        model=model,
        num_searches=NUM_SEARCHES,
        c_puct=C_PUCT,
        batch_size=MCTS_BATCH_SIZE,
        dirichlet_alpha=0.0,  # no exploration noise at play time
    )

## 3. Board rendering

Plain text. `X` is whoever moves first, `O` is the opponent, `.` is empty. Column indices are printed at the bottom so you can pick a move.

In [16]:
def render(state, human_player):
    """Print the board with the human's piece always shown as X."""
    sym = {human_player: "X", -human_player: "O", 0: "."}
    print()
    for row in state:
        print(" " + " ".join(sym[int(v)] for v in row))
    print(" " + " ".join(str(c) for c in range(game.column_count)))
    print()


def agent_move(mcts, state, player, show_policy=True):
    """Run MCTS, optionally show the visit-count distribution, return chosen action."""
    policy = mcts.search(state, player)
    action = int(np.argmax(policy))
    if show_policy:
        bar = "  ".join(
            f"{c}:{policy[c]:.2f}" for c in range(game.action_size) if policy[c] > 0
        )
        print(f"  agent thinks: {bar}")
    return action

## 4. Play a game

`play(human_first=True)` lets you go first (you'll be `X`). Pass `human_first=False` to make the agent move first.

In [17]:
def play(human_first=True, show_policy=True):
    mcts = make_mcts()
    state = game.get_initial_state()
    human_player = 1 if human_first else -1
    current = 1  # whoever moves first is +1 by convention

    while True:
        render(state, human_player)
        valid = game.get_valid_moves(state)
        legal_cols = [c for c in range(game.action_size) if valid[c]]

        if current == human_player:
            while True:
                raw = input(f"Your move (cols {legal_cols}, q to quit): ").strip()
                if raw.lower() in ("q", "quit", "exit"):
                    print("Aborted.")
                    return
                try:
                    action = int(raw)
                except ValueError:
                    print("  not a number, try again")
                    continue
                if action not in legal_cols:
                    print(f"  column {action} not playable")
                    continue
                break
        else:
            action = agent_move(mcts, state, current, show_policy=show_policy)
            print(f"  agent plays column {action}")

        # Both sides advance the MCTS root so tree reuse works for the agent's turn
        mcts.advance_root(action)
        state = game.update_state(state, action, current)
        value, terminated = game.get_value_and_terminated(state, action)
        if terminated:
            render(state, human_player)
            if value == 0:
                print("Draw.")
            elif current == human_player:
                print("You win.")
            else:
                print("Agent wins.")
            return
        current = game.get_opponent(current)

In [ ]:
play(human_first=True)


 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 0 1 2 3 4 5 6



Your move (cols [0, 1, 2, 3, 4, 5, 6], q to quit):  3



 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . X . . .
 0 1 2 3 4 5 6

  agent thinks: 0:0.02  1:0.10  2:0.22  3:0.58  4:0.04  5:0.02  6:0.02
  agent plays column 3

 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . O . . .
 . . . X . . .
 0 1 2 3 4 5 6



Your move (cols [0, 1, 2, 3, 4, 5, 6], q to quit):  2



 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . O . . .
 . . X X . . .
 0 1 2 3 4 5 6

  agent thinks: 0:0.02  1:0.88  2:0.02  3:0.02  4:0.02  5:0.02  6:0.02
  agent plays column 1

 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . O . . .
 . O X X . . .
 0 1 2 3 4 5 6



...


 . . . . . . .
 . . X . . O .
 . . O . X O O
 . . O . O X X
 . . O O X X X
 . O X X X O X
 0 1 2 3 4 5 6

  agent thinks: 0:0.04  1:0.10  2:0.12  3:0.57  4:0.06  5:0.06  6:0.06
  agent plays column 3

 . . . . . . .
 . . X . . O .
 . . O . X O O
 . . O O O X X
 . . O O X X X
 . O X X X O X
 0 1 2 3 4 5 6



Your move (cols [0, 1, 2, 3, 4, 5, 6], q to quit):  


  not a number, try again


Your move (cols [0, 1, 2, 3, 4, 5, 6], q to quit):  3



 . . . . . . .
 . . X . . O .
 . . O X X O O
 . . O O O X X
 . . O O X X X
 . O X X X O X
 0 1 2 3 4 5 6

  agent thinks: 0:0.03  1:0.05  2:0.05  3:0.70  4:0.05  5:0.06  6:0.08
  agent plays column 3

 . . . . . . .
 . . X O . O .
 . . O X X O O
 . . O O O X X
 . . O O X X X
 . O X X X O X
 0 1 2 3 4 5 6



## 5. Optional: agent vs agent

Quick sanity check — watch the model play itself. With a trained model, expect first-player wins (Connect 4 is a first-player-wins game with perfect play); with an untrained model, expect noisy junk.

In [8]:
def watch_self_play(show_policy=False):
    mcts1 = make_mcts()
    mcts2 = make_mcts()
    state = game.get_initial_state()
    current = 1
    while True:
        render(state, human_player=1)
        mcts = mcts1 if current == 1 else mcts2
        action = agent_move(mcts, state, current, show_policy=show_policy)
        print(f"  player {current:+d} plays column {action}")
        mcts1.advance_root(action)
        mcts2.advance_root(action)
        state = game.update_state(state, action, current)
        value, terminated = game.get_value_and_terminated(state, action)
        if terminated:
            render(state, human_player=1)
            print("Draw." if value == 0 else f"Player {current:+d} wins.")
            return
        current = game.get_opponent(current)


watch_self_play()


 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 0 1 2 3 4 5 6

  player +1 plays column 2

 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . X . . . .
 0 1 2 3 4 5 6

  player -1 plays column 3

 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . X O . . .
 0 1 2 3 4 5 6

  player +1 plays column 3

 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . X . . .
 . . X O . . .
 0 1 2 3 4 5 6

  player -1 plays column 3

 . . . . . . .
 . . . . . . .
 . . . . . . .
 . . . O . . .
 . . . X . . .
 . . X O . . .
 0 1 2 3 4 5 6

  player +1 plays column 3

 . . . . . . .
 . . . . . . .
 . . . X . . .
 . . . O . . .
 . . . X . . .
 . . X O . . .
 0 1 2 3 4 5 6

  player -1 plays column 3

 . . . . . . .
 . . . O . . .
 . . . X . . .
 . . . O . . .
 . . . X . . .
 . . X O . . .
 0 1 2 3 4 5 6

  player +1 plays column 3

 . . . X . . .
 . . . O . . .
 . . . X . . .
 . . . O . . .
 